In [1]:
import numpy as np
import polars as pl

In [2]:
pl.__version__

'1.36.1'

In [3]:
!HF_ENDPOINT="http://huggingface.proxy" hf download deepvk/VK-LSVD --repo-type dataset --include "metadata/*" --local-dir  /home/jovyan/vkml/data/vk_lsvd/raw

Fetching 3 files: 100%|███████████████████████████| 3/3 [00:08<00:00,  2.87s/it]
Download complete: 100%|████████████████████| 2.59G/2.59G [00:08<00:00, 578MB/s]/home/jovyan/vkml/data/vk_lsvd/raw
Download complete: 100%|████████████████████| 2.59G/2.59G [00:08<00:00, 300MB/s]


In [4]:
!HF_ENDPOINT="http://huggingface.proxy" hf download deepvk/VK-LSVD --repo-type dataset --include "subsamples/ur0.01_ir0.01/*" --local-dir /home/jovyan/vkml/data/vk_lsvd/raw

Fetching 27 files: 100%|████████████████████████| 27/27 [00:07<00:00,  3.73it/s]
Download complete: 100%|███████████████████| 25.8M/25.8M [00:07<00:00, 5.25MB/s]/home/jovyan/vkml/data/vk_lsvd/raw
Download complete: 100%|███████████████████| 25.8M/25.8M [00:07<00:00, 3.56MB/s]


Разбиение сабсэмплов на базовую, гэп, вал и тест части

Добавляется колонка original_order чтобы сохранять порядок внутри каждой из частей

In [47]:
subsample_name = 'ur0.01_ir0.01'

DATASET_PATH = "/home/jovyan/vkml/data/vk_lsvd/raw"

metadata_files = ['metadata/users_metadata.parquet',
                  'metadata/items_metadata.parquet',
                  'metadata/item_embeddings.npz']

BASE_WEEKS = (0, 21)
GAP_WEEKS = (21, 23)
VAL_WEEKS = (23, 24)
TEST_WEEKS = (24, 25)

base_interactions_files = [f'subsamples/{subsample_name}/train/week_{i:02}.parquet'
                            for i in range(BASE_WEEKS[0], BASE_WEEKS[1])]

gap_interactions_files = [f'subsamples/{subsample_name}/train/week_{i:02}.parquet'
                            for i in range(GAP_WEEKS[0], GAP_WEEKS[1])]

val_interactions_files = [f'subsamples/{subsample_name}/train/week_{i:02}.parquet'
                            for i in range(VAL_WEEKS[0], VAL_WEEKS[1])]

test_interactions_files = [f'subsamples/{subsample_name}/train/week_{i:02}.parquet'
                            for i in range(TEST_WEEKS[0], TEST_WEEKS[1])]

all_interactions_files = base_interactions_files + gap_interactions_files + val_interactions_files + test_interactions_files

base_with_gap_interactions_files = base_interactions_files + gap_interactions_files

len(base_interactions_files), len(gap_interactions_files), len(val_interactions_files), len(test_interactions_files), len(all_interactions_files), len(base_with_gap_interactions_files)

(21, 2, 1, 1, 25, 23)

In [48]:
def get_parquet_interactions(data_files, positive_event_timespent):
    data_interactions = pl.concat([pl.scan_parquet(f'{DATASET_PATH}/{file}')
                                for file in data_files])
    data_interactions = data_interactions.collect()
    data_interactions = data_interactions.with_row_index("original_order")
    data_interactions = data_interactions.filter(pl.col('timespent') > positive_event_timespent)
    return data_interactions


In [49]:
POSITIVE_EVENT_TIMESPENT = 15
base_interactions = get_parquet_interactions(base_interactions_files, POSITIVE_EVENT_TIMESPENT)
gap_interactions = get_parquet_interactions(gap_interactions_files, POSITIVE_EVENT_TIMESPENT)
val_interactions = get_parquet_interactions(val_interactions_files, POSITIVE_EVENT_TIMESPENT)
test_interactions = get_parquet_interactions(test_interactions_files, POSITIVE_EVENT_TIMESPENT)
all_data_interactions = get_parquet_interactions(all_interactions_files, POSITIVE_EVENT_TIMESPENT)
base_with_gap_interactions = get_parquet_interactions(base_with_gap_interactions_files, POSITIVE_EVENT_TIMESPENT)

Загрузка и фильтрация эмбеддингов

In [50]:
all_data_users = all_data_interactions.select('user_id').unique()
all_data_items = all_data_interactions.select('item_id').unique()

item_ids = np.load(f"{DATASET_PATH}/metadata/item_embeddings.npz")['item_id']
item_embeddings = np.load(f"{DATASET_PATH}/metadata/item_embeddings.npz")['embedding']

mask = np.isin(item_ids, all_data_items.to_numpy())
item_ids = item_ids[mask]
item_embeddings = item_embeddings[mask]

items_metadata = pl.read_parquet(f"{DATASET_PATH}/metadata/items_metadata.parquet")

items_metadata = items_metadata.join(all_data_items, on='item_id')
items_metadata = items_metadata.join(pl.DataFrame({'item_id': item_ids, 
                                                   'embedding': item_embeddings}), on='item_id')

only_base_items_metadata = items_metadata.join(base_interactions.select('item_id').unique(), on='item_id')
only_base_with_gap_items_metadata = items_metadata.join(base_with_gap_interactions.select('item_id').unique(), on='item_id')

Сжатие айтем айди и ремапинг

In [51]:
all_data_items = all_data_interactions.select('item_id').unique()
all_data_users = all_data_interactions.select('user_id').unique()

unique_items_sorted = all_data_items.sort('item_id').with_row_index('new_item_id')
global_item_mapping = dict(zip(unique_items_sorted['item_id'], unique_items_sorted['new_item_id']))

print(f"Total users: {all_data_users.shape[0]}, Total items: {len(global_item_mapping)}")

Total users: 77403, Total items: 65659


In [52]:
def remap_interactions(df, mapping): 
    return df.with_columns( 
        pl.col('item_id') 
        .map_elements(lambda x: mapping.get(x, None), return_dtype=pl.UInt32)
    ) 

base_interactions_remapped = remap_interactions(base_interactions, global_item_mapping) 
gap_interactions_remapped = remap_interactions(gap_interactions, global_item_mapping) 
test_interactions_remapped = remap_interactions(test_interactions, global_item_mapping) 
val_interactions_remapped = remap_interactions(val_interactions, global_item_mapping) 
all_data_interactions_remapped = remap_interactions(all_data_interactions, global_item_mapping) 
base_with_gap_interactions_remapped = remap_interactions(base_with_gap_interactions, global_item_mapping)

items_metadata_remapped = remap_interactions(items_metadata, global_item_mapping) 
only_base_items_metadata_remapped = remap_interactions(only_base_items_metadata, global_item_mapping) 
only_base_with_gap_items_metadata_remapped = remap_interactions(only_base_with_gap_items_metadata, global_item_mapping)

Группировка по юзер айди

In [53]:
def get_grouped_interactions(data_interactions: pl.DataFrame):
    print(f"interactions count: {data_interactions.shape}")
    data_interactions = data_interactions.sort(by='original_order')

    data_res = (
        data_interactions
        .select(['original_order', 'user_id', 'item_id'])
        .group_by('user_id', maintain_order=True)
        .agg(
            pl.col('item_id')
            .sort_by('original_order')
            .alias('item_ids'),
            
            pl.col('original_order')
            .sort()
            .alias('timestamps'),
        )
        .rename({'user_id': 'uid'})
        .sort('uid')
    )
    
    print(f"users count: {data_res.shape}")
    return data_res


base_interactions_grouped = get_grouped_interactions(base_interactions_remapped)
gap_interactions_grouped = get_grouped_interactions(gap_interactions_remapped)
test_interactions_grouped = get_grouped_interactions(test_interactions_remapped)
val_interactions_grouped = get_grouped_interactions(val_interactions_remapped)
all_data_interactions_grouped = get_grouped_interactions(all_data_interactions_remapped)
base_with_gap_interactions_grouped = get_grouped_interactions(base_with_gap_interactions_remapped)

# 1 week gap 15ts
# interactions count: (1206762, 13)
# users count: (75281, 3)
# interactions count: (66879, 13)
# users count: (28610, 3)
# interactions count: (69887, 13)
# users count: (28982, 3)
# interactions count: (73637, 13)
# users count: (30231, 3)
# interactions count: (1417165, 13)
# users count: (77403, 3)

# 2 week gap 15ts
# interactions count: (1138730, 13)
# users count: (74588, 3)
# interactions count: (134911, 13)
# users count: (39437, 3)
# interactions count: (69887, 13)
# users count: (28982, 3)
# interactions count: (73637, 13)
# users count: (30231, 3)
# interactions count: (1417165, 13)
# users count: (77403, 3)

# 3 week gap 15ts
# interactions count: (1081689, 13)
# users count: (73910, 3)
# interactions count: (191952, 13)
# users count: (45400, 3)
# interactions count: (69887, 13)
# users count: (28982, 3)
# interactions count: (73637, 13)
# users count: (30231, 3)
# interactions count: (1417165, 13)
# users count: (77403, 3)

interactions count: (1138730, 13)
users count: (74588, 3)
interactions count: (134911, 13)
users count: (39437, 3)
interactions count: (69887, 13)
users count: (28982, 3)
interactions count: (73637, 13)
users count: (30231, 3)
interactions count: (1417165, 13)
users count: (77403, 3)
interactions count: (1273641, 13)
users count: (76011, 3)


In [54]:
base_interactions_grouped.head(1)

uid,item_ids,timestamps
u32,list[u32],list[u32]
59,"[23108, 39641, 43801]","[441788, 850393, 3006839]"


Сохранение

In [56]:
import json
OUTPUT_DIR = f"/home/jovyan/vkml/data/vk_lsvd/{len(gap_interactions_files)}w-gap-{POSITIVE_EVENT_TIMESPENT}-ts"
OUTPUT_DIR

import os
os.mkdir(OUTPUT_DIR)

In [57]:
mapping_output_path = f"{OUTPUT_DIR}/global_item_mapping.json"

with open(mapping_output_path, 'w') as f:
    json.dump({str(k): v for k, v in global_item_mapping.items()}, f, indent=2)

print(f"Сохранён маппинг: {mapping_output_path}")

Сохранён маппинг: /home/jovyan/vkml/data/vk_lsvd/2w-gap-15-ts/global_item_mapping.json


In [58]:
def write_parquet(output_dir, data, file_name):
    print(f"размерность: {data.shape}")
    output_parquet_path = f"{output_dir}/{file_name}.parquet"
    data.write_parquet(output_parquet_path)
    print(f"Сохранен файл: {file_name}")

write_parquet(OUTPUT_DIR, items_metadata_remapped, "items_metadata_remapped")
write_parquet(OUTPUT_DIR, items_metadata, "items_metadata_old")

write_parquet(OUTPUT_DIR, only_base_items_metadata_remapped, "only_base_items_metadata_remapped")
write_parquet(OUTPUT_DIR, only_base_items_metadata, "only_base_items_metadata_old")

write_parquet(OUTPUT_DIR, only_base_with_gap_items_metadata_remapped, "only_base_with_gap_items_metadata_remapped")
write_parquet(OUTPUT_DIR, only_base_with_gap_items_metadata, "only_base_with_gap_items_metadata_old")

write_parquet(OUTPUT_DIR, base_interactions_grouped, "base_interactions_grouped")
write_parquet(OUTPUT_DIR, gap_interactions_grouped, "gap_interactions_grouped")
write_parquet(OUTPUT_DIR, test_interactions_grouped, "test_interactions_grouped")
write_parquet(OUTPUT_DIR, val_interactions_grouped, "val_interactions_grouped")
write_parquet(OUTPUT_DIR, base_with_gap_interactions_grouped, "base_with_gap_interactions_grouped")

write_parquet(OUTPUT_DIR, all_data_interactions_grouped, "all_data_interactions_grouped")

write_parquet(OUTPUT_DIR, all_data_interactions_remapped, "all_data_interactions_remapped")

размерность: (65659, 5)
Сохранен файл: items_metadata_remapped
размерность: (65659, 5)
Сохранен файл: items_metadata_old
размерность: (58668, 5)
Сохранен файл: only_base_items_metadata_remapped
размерность: (58668, 5)
Сохранен файл: only_base_items_metadata_old
размерность: (62362, 5)
Сохранен файл: only_base_with_gap_items_metadata_remapped
размерность: (62362, 5)
Сохранен файл: only_base_with_gap_items_metadata_old
размерность: (74588, 3)
Сохранен файл: base_interactions_grouped
размерность: (39437, 3)
Сохранен файл: gap_interactions_grouped
размерность: (28982, 3)
Сохранен файл: test_interactions_grouped
размерность: (30231, 3)
Сохранен файл: val_interactions_grouped
размерность: (76011, 3)
Сохранен файл: base_with_gap_interactions_grouped
размерность: (77403, 3)
Сохранен файл: all_data_interactions_grouped
размерность: (1417165, 13)
Сохранен файл: all_data_interactions_remapped
